In [ ]:
import os
import ray
import numpy as np
import pandas as pd
import logging
from ray import tune

from ray.tune.schedulers.pb2 import PB2, PopulationBasedTraining
from ray.tune import Checkpoint, run, sample_from 

from importnb import Notebook
with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv
    from LabUserRequest import UserRequestEvents
    from LabEnvWrapperKunkali import EnvWrapper

import os
import sys

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

import Common.config as config
import Common.datatypes as datatypes
import Common.debugger as debugger
import Common.utils as utils

import importlib

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(debugger)
importlib.reload(utils)

In [ ]:
cfg = config.Config()

In [ ]:
class PGTrainer(object):
    def __init__(self, env, model, optimizer, scheduler, gamma=0.99, update_target_every=10):
        self.env = env
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.gamma = gamma
        self.update_target_every = update_target_every
        self.n_step_buffer = []

In [ ]:
def rl(cfg):

    alg_name ="ATLA"
    env_name ="EVS"

    trainer = PGTrainer(args, model, env, logger, constraint_model)

    for i in range(cfg.n_episodes):
        stat = {}
        hyperparams = {
            "n_step": 3,
            "gamma": 0.99,
            "lr": 1e-4,
            "batch_size": 64,
            "buffer_capacity": 10000
        }
        trainer.run(stat, i, hyperparams=hyperparams)

In [ ]:
if __name__ == "__main__":
    pb2 = PB2(
        metric="objective",
        mode="max",
        quantile_fraction=0.25,
        perturbation_interval=160,
        hyperparam_bounds={
            "alpha": [0.05, 0.2],
            "beta": [0.3, 0.6]
        }
    )
    
    for seed in range(0, 4):
        analysis = tune.run(
            rl,
            scheduler=pb2,
            num_samples=4,
            reuse_actors=True,
            config={
                "alpha": sample_from(lambda spec: np.random.uniform(0.05, 0.2)),
                "beta": sample_from(lambda spec: np.random.uniform(0.4, 0.6)),
                "seed": seed
            }
        )
        
        all_dfs = analysis.trial_dataframes
        names = list(all_dfs.keys())
        
        results = pd.DataFrame()
        for i in range(4):
            df = all_dfs[names[i]].copy()
            df['sample_num'] = i 
            results = pd.concat([results, df]).reset_index(drop=True)

        dir = "{}_{}_{}_Size{}_{}_{}_{}_{}_{}".format(rl, "file", "method", str(4), "env", "default", "max", "160", "batch")
        exist_dir = os.path.expanduser('~/data/' + dir)
        if not(os.path.exists(exist_dir)):
            os.makedirs(exist_dir)

        result_dir1 = os.path.expanduser('~/data/')
        result_dir2 = f"{dir}/seed{seed}.csv"
        results.to_csv(result_dir1 + result_dir2)
        